# 20260919 Neural Preprocess

Dated pipeline notebook for checking one recording at a time: AVI-to-H5 conversion, R2025b EXTRACT, R2021b ActSort/manualActSort curation, and curated-neuron import.

# Part 1: AVI to H5

These cells come from the updated AVI-to-H5 workflow.

# AVI to H5 Pipeline Check

Use this notebook to inspect one miniscope video from the spreadsheet before running the batch converter.

In [ ]:
from pathlib import Path
import os
import sys

import h5py
import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.pipeline import default_neu_h5_path
from preprocess_functions.video import check_leading_video_corruption, convert_avi_to_h5


In [ ]:
MANIFEST = Path("data_paths/RSC_PPC_Cohort1_paths.xlsx")
SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
OUTPUT_ROOT = Path("preprocess_out")

# Paste a recording_id from the table below, or leave as None for the first row with neu_vid.
RECORDING_ID = None


In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "trial_type": record.trial_type,
        "neu_vid": str(record.neu_vid) if record.neu_vid else None,
        "local_h5": str(default_neu_h5_path(OUTPUT_ROOT, record)),
    }
    for record in records
])
records_df


In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        for record in records:
            if record.neu_vid is not None:
                return record
        raise ValueError("No rows with a neu_vid path were found.")

    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")

record = choose_record(records, RECORDING_ID)
h5_path = default_neu_h5_path(OUTPUT_ROOT, record)
print("recording_id:", record.recording_id)
print("source AVI:", record.neu_vid)
print("local H5:", h5_path)


## Check One Video

The checker only identifies bad leading frames. During conversion those frames become copied-frame gaps, so the H5 keeps the original frame count for neural/behavior alignment.

In [ ]:
report = check_leading_video_corruption(record.neu_vid)
pd.Series({
    "status": report["status"],
    "leading_bad_frames": report["leading_bad_frames"],
    "first_clean_frame": report["first_clean_frame"],
    "clean_run_found": report["clean_run_found"],
    "n_suspicious_later_frames": report["n_suspicious_later_frames"],
    "metadata_warnings": ";".join(report["metadata_warnings"]),
})


## Convert This Recording

This writes to `preprocess_out/<recording_id>/neural/` by default. Change `OVERWRITE` only when you intentionally want to rebuild the H5.

In [ ]:
OVERWRITE = False
summary = convert_avi_to_h5(
    record.neu_vid,
    h5_path,
    overwrite=OVERWRITE,
    gap_leading_corrupt=True,
    gap_fill_strategy="nearest",
)
pd.Series(summary)


In [ ]:
with h5py.File(h5_path, "r") as h5f:
    h5_summary = {
        "data_shape": h5f["data"].shape,
        "valid_frame_mask_shape": h5f["valid_frame_mask"].shape,
        "frame_count": h5f.attrs.get("frame_count"),
        "valid_frame_count": h5f.attrs.get("valid_frame_count"),
        "gapped_leading_frames": h5f.attrs.get("gapped_leading_frames"),
        "gap_fill_strategy": h5f.attrs.get("gap_fill_strategy"),
    }
pd.Series(h5_summary)


## Batch Command

After one video looks right, run the script over the spreadsheet. Use `--only RECORDING_ID` first, then remove it for the full batch.

In [ ]:
print(
    "python scripts/convert_miniscope_avi_to_h5.py "
    f"{MANIFEST} --output-root {OUTPUT_ROOT} --only {record.recording_id} --overwrite"
)


## Next: R2025b EXTRACT

Once the H5 exists, run EXTRACT in MATLAB R2025b. ActSort/manualActSort comes later in MATLAB R2021b.

In [ ]:
matlab_r2025b = r"C:\Program Files\MATLAB\R2025b\bin\matlab.exe"
print(
    f'python scripts/run_matlab_neural_extraction.py "{OUTPUT_ROOT / "manifest_with_h5.csv"}" '
    '--matlab-script matlab/run_extract_template.m '
    f'--matlab-bin "{matlab_r2025b}" '
    f'--output-root "{OUTPUT_ROOT}" '
    f'--only "{record.recording_id}"'
)


# Part 2: EXTRACT, ActSort, and Curated Neuron Import

These cells come from the updated neural trace import workflow.

# Import Curated EXTRACT/ActSort Neurons

Use this after MATLAB R2025b has run Schnitzer lab EXTRACT and MATLAB R2021b has saved the ActSort/manualActSort labels.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from preprocess_functions.manifest import load_session_records
from preprocess_functions.neural import find_extract_outputs, import_extract_curated_cells
from preprocess_functions.pipeline import (
    default_cell_csv_path,
    default_curated_neurons_path,
    matlab_output_dir,
)


In [ ]:
OUTPUT_ROOT = Path("preprocess_out")
MANIFEST = OUTPUT_ROOT / "manifest_with_matlab.csv"
if not MANIFEST.exists():
    MANIFEST = OUTPUT_ROOT / "manifest_with_h5.csv"

SHEET = None
LAB_DRIVE = os.environ.get("LAB_DRIVE_PATH") or None
TRACE_KIND = "raw"
RECORDING_ID = None

# GPU/EXTRACT MATLAB and ActSort MATLAB are intentionally separate.
MATLAB_R2025B = r"C:\Program Files\MATLAB\R2025b\bin\matlab.exe"
MATLAB_R2021B = r"C:\Program Files\MATLAB\R2021b\bin\matlab.exe"


In [ ]:
records = load_session_records(MANIFEST, sheet_name=SHEET, lab_drive=LAB_DRIVE)
records_df = pd.DataFrame([
    {
        "recording_id": record.recording_id,
        "session_id": record.session_id,
        "trial_type": record.trial_type,
        "matlab_output_dir": str(record.matlab_output_dir or matlab_output_dir(OUTPUT_ROOT, record)),
        "cell_csv": str(record.cell_csv or default_cell_csv_path(OUTPUT_ROOT, record, trace_kind=TRACE_KIND)),
    }
    for record in records
])
records_df


In [ ]:
def choose_record(records, recording_id=None):
    if recording_id is None:
        return records[0]
    for record in records:
        if recording_id in {record.recording_id, record.session_id, record.trial_type, record.mouse_id}:
            return record
    raise ValueError(f"No manifest row matched {recording_id!r}.")

record = choose_record(records, RECORDING_ID)
out_dir = record.matlab_output_dir or matlab_output_dir(OUTPUT_ROOT, record)
cell_csv = record.cell_csv or default_cell_csv_path(OUTPUT_ROOT, record, trace_kind=TRACE_KIND)
curated_csv = default_curated_neurons_path(OUTPUT_ROOT, record)

print("recording_id:", record.recording_id)
print("MATLAB output dir:", out_dir)
print("cell trace CSV:", cell_csv)
print("curated neuron CSV:", curated_csv)


## Run EXTRACT in MATLAB R2025b

Run this after the AVI-to-H5 conversion. R2025b should have EXTRACT-public and GPU support on the MATLAB path.

In [ ]:
extract_manifest = OUTPUT_ROOT / "manifest_with_h5.csv"
extract_cmd = (
    f'python scripts/run_matlab_neural_extraction.py "{extract_manifest}" '
    f'--matlab-script matlab/run_extract_template.m '
    f'--matlab-bin "{MATLAB_R2025B}" '
    f'--output-root "{OUTPUT_ROOT}" '
    f'--only "{record.recording_id}"'
)
print(extract_cmd)


## Or Loop Inside MATLAB R2025b

If you are already inside MATLAB on the GPU PC, this avoids launching MATLAB once per recording.

In [ ]:
print("addpath('matlab')")
print("run_extract_batch_from_index( ...")
print("    'preprocess_out/manifest_with_h5.csv', ...")
print("    'preprocess_out', ...")
print(f"    'Only', '{record.recording_id}')")


## Curate in MATLAB R2021b

Open R2021b for ActSort/manualActSort. Load the unsorted EXTRACT output, classify accepted cells, and save the labels file in the same MATLAB output folder.

In [ ]:
unsorted_mat = out_dir / f"{record.recording_id}_extract_output_unsorted.mat"
precomputed_mat = out_dir / f"{record.recording_id}_precomputed_output.mat"
labels_mat = out_dir / f"{record.recording_id}_precomputed_output_LABELS.mat"

print("Open this in MATLAB R2021b / ActSort:")
print(unsorted_mat)
print()
print("EXTRACT output expected by Python:")
print(precomputed_mat)
print()
print("Save ActSort/manualActSort labels here:")
print(labels_mat)


## Find MATLAB Outputs

Expected files are `<recording_id>_precomputed_output.mat` from R2025b EXTRACT and `<recording_id>_precomputed_output_LABELS.mat` from R2021b ActSort/manualActSort.

In [ ]:
traces_mat, labels_mat = find_extract_outputs(
    out_dir,
    recording_id=record.recording_id,
    session_id=record.session_id,
)
print("EXTRACT traces:", traces_mat)
print("ActSort labels:", labels_mat)


## Import Accepted Cells

ActSort/manualActSort label `1` is treated as accepted. Set `TRACE_KIND = "deconvolved"` if you want the OASIS deconvolved traces instead of raw accepted traces.

In [ ]:
summary = import_extract_curated_cells(
    traces_mat_path=traces_mat,
    labels_mat_path=labels_mat,
    output_csv=cell_csv,
    curated_csv=curated_csv,
    recording_id=record.recording_id,
    session_id=record.session_id,
    trace_kind=TRACE_KIND,
)
pd.Series(summary)


In [ ]:
cell_df = pd.read_csv(cell_csv)
curated_df = pd.read_csv(curated_csv)
print(cell_df.shape)
display(cell_df.head())
display(curated_df.head())


## Batch Command

Once the one-recording import works, use the script to import the rest of the spreadsheet.

In [ ]:
print(
    "python scripts/import_curated_neurons.py "
    f"{MANIFEST} --output-root {OUTPUT_ROOT} --trace-kind {TRACE_KIND} --only {record.recording_id}"
)
